# DVF data exploration

Load a DVF file from `data/raw/` and explore mutations (nature, date, value).

In [1]:
from pathlib import Path
import sys

# Ensure project src is on path (kernel cwd is often the notebook dir, not project root)
_project_root = Path.cwd() if (Path.cwd() / "src" / "dvf").exists() else Path.cwd().parent
_src = _project_root / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from dvf import load_dvf_raw, load_dvf_csv, load_dvf_plus, summarize_mutations

In [2]:
# Point to a file in data/raw (pipe-separated .txt or CSV)
_project_root = Path.cwd() if (Path.cwd() / "src" / "dvf").exists() else Path.cwd().parent
raw_dir = _project_root / "data" / "raw"
files = list(raw_dir.glob("*"))
print("Files in data/raw:", [f.name for f in files if not f.name.startswith(".")])

Files in data/raw: ['mutations_d13.csv']


In [3]:
# Load a sample (use nrows for large files)
# df = load_dvf_raw(raw_dir / "dvf_2023.txt", nrows=50_000)
# DVF+ files are pipe-separated: use load_dvf_plus (not load_dvf_csv) to avoid parser warnings
df = load_dvf_plus(raw_dir / "mutations_d13.csv", nrows=50_000, sep=";")
df.head()

,idmutation,idmutinvar,idopendata,idnatmut,codservch,refdoc,datemut,anneemut,moismut,coddep,...,sapt3pp,sapt4pp,sapt5pp,smai1pp,smai2pp,smai3pp,smai4pp,smai5pp,codtypbien,libtypbien
0,1187708,5c0368256b672fc53466f1c6060fb8ea,5c0368256b672fc53466f1c6060fb8ea,1,NaN,NaN,2024-04-23,2024,4,13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2313,TERRAIN DE TYPE TERRE ET PRE
1,1183782,89caa38ae54f9e893cc0d51a05260919,89caa38ae54f9e893cc0d51a05260919,6,NaN,NaN,2024-06-03,2024,6,13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,229,TERRAIN ARTIFICIALISE MIXTE
2,1184799,1f164562fedfb0611543e317df087218,1f164562fedfb0611543e317df087218,1,NaN,NaN,2024-06-07,2024,6,13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,121,UN APPARTEMENT
3,1184849,265c192a0a020b6343d97ebc2c524659,265c192a0a020b6343d97ebc2c524659,1,NaN,NaN,2024-04-30,2024,4,13,...,0.0,0.0,0.0,0.0,0.0,0.0,85.0,0.0,111,UNE MAISON
4,1185046,57989a3bfead4cc1edbb5eb49b706a6c,57989a3bfead4cc1edbb5eb49b706a6c,1,NaN,NaN,2024-06-21,2024,6,13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,233,TERRAIN LANDES ET EAUX


In [4]:
# Distinct values of libtypbien (property type) and their counts
df["libtypbien"].value_counts()

libtypbien
UN APPARTEMENT                               22754
UNE MAISON                                    9986
BATI - INDETERMINE : Vefa sans descriptif     3730
UNE DEPENDANCE                                3712
ACTIVITE                                      2420
DEUX APPARTEMENTS                              975
TERRAIN DE TYPE TAB                            929
TERRAIN ARTIFICIALISE MIXTE                    900
TERRAIN DE TYPE TERRE ET PRE                   851
BATI MIXTE - LOGEMENT/ACTIVITE                 849
DES DEPENDANCES                                500
APPARTEMENT INDETERMINE                        432
DES MAISONS                                    392
TERRAIN VERGER                                 281
TERRAIN LANDES ET EAUX                         277
TERRAIN NON BATIS INDETERMINE                  258
TERRAIN FORESTIER                              204
BATI - INDETERMINE : Vente avec volume(s)      187
BATI MIXTE - LOGEMENTS                         160
TERRAIN VITICOLE    

In [5]:
# Filter: houses only, commune INSEE 13033 (Ensues)
# l_codinsee is stored as "['13033']" in mutations_d13.csv, so use .str.contains()
df = df[
    (df["libtypbien"] == "UNE MAISON")
    & (df["l_codinsee"].astype(str).str.contains("13033", na=False))
]
df.head()

,idmutation,idmutinvar,idopendata,idnatmut,codservch,refdoc,datemut,anneemut,moismut,coddep,...,sapt3pp,sapt4pp,sapt5pp,smai1pp,smai2pp,smai3pp,smai4pp,smai5pp,codtypbien,libtypbien
5901,1188277,32742268f7659751972a6a3f9af70507,32742268f7659751972a6a3f9af70507,1,NaN,NaN,2024-02-13,2024,2,13,...,0.0,0.0,0.0,0.0,50.0,0.0,0.0,0.0,111,UNE MAISON
6732,1189111,e2ae3ecaa9a6e4bcae52f51d6df83e9a,e2ae3ecaa9a6e4bcae52f51d6df83e9a,1,NaN,NaN,2024-06-10,2024,6,13,...,0.0,0.0,0.0,0.0,0.0,0.0,106.0,0.0,111,UNE MAISON
9404,1191807,55ac2022d2799f9590d47627995cadf5,55ac2022d2799f9590d47627995cadf5,1,NaN,NaN,2024-01-31,2024,1,13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,134.0,111,UNE MAISON
9633,1192042,79b820e9c276ae7e5e2ba517b1deb0f2,79b820e9c276ae7e5e2ba517b1deb0f2,1,NaN,NaN,2024-01-23,2024,1,13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,152.0,111,UNE MAISON
11576,1194000,acfb6dcce5fb1f1681778234cc4fe28a,acfb6dcce5fb1f1681778234cc4fe28a,1,NaN,NaN,2024-01-23,2024,1,13,...,0.0,0.0,0.0,44.0,0.0,0.0,0.0,0.0,111,UNE MAISON


In [7]:
df.columns

Index(['idmutation', 'idmutinvar', 'idopendata', 'idnatmut', 'codservch',
       'refdoc', 'datemut', 'anneemut', 'moismut', 'coddep', 'libnatmut',
       'vefa', 'valeurfonc', 'nbdispo', 'nblot', 'nbcomm', 'l_codinsee',
       'nbsection', 'l_section', 'nbpar', 'l_idpar', 'nbparmut', 'l_idparmut',
       'nbsuf', 'sterr', 'nbvolmut', 'nblocmut', 'l_idlocmut', 'nblocmai',
       'nblocapt', 'nblocdep', 'nblocact', 'nbapt1pp', 'nbapt2pp', 'nbapt3pp',
       'nbapt4pp', 'nbapt5pp', 'nbmai1pp', 'nbmai2pp', 'nbmai3pp', 'nbmai4pp',
       'nbmai5pp', 'sbati', 'sbatmai', 'sbatapt', 'sbatact', 'sapt1pp',
       'sapt2pp', 'sapt3pp', 'sapt4pp', 'sapt5pp', 'smai1pp', 'smai2pp',
       'smai3pp', 'smai4pp', 'smai5pp', 'codtypbien', 'libtypbien'],
      dtype='object')